In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *


In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
file_hmi = files[0]
file_fdt = files[1]

file_hmi, file_fdt

('/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz')

In [6]:
cor = np.zeros((20,20))

for i, dcrota in enumerate(np.arange(0.2,0.4,0.01)):
    for j, drsun in enumerate(np.arange(0.1,0.5,0.02)):

        with fits.open(file_hmi) as hdul:
            header_hmi = hdul[1].header
            data_hmi = hdul[1].data

        with fits.open(file_fdt) as hdul:
            header_fdt = hdul[0].header
            data_fdt = hdul[0].data

        data_fdt = undistort(data_fdt, header_fdt, xd, yd)

        data_hmi = rebin(data_hmi, 12, update_header=header_hmi)
        data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=True, mu_thr=0.7, crota=header_fdt['CROTA'] + dcrota, rsun=330.25 + drsun, xc=421.64, yc=457.87)

        data_fdt[np.isnan(data_hmi)] = np.nan

        q = np.nanmean(data_fdt * data_hmi) / np.sqrt(np.nanmean(data_fdt ** 2) * np.nanmean(data_hmi ** 2))
        cor[i,j] = q

        print(dcrota, drsun, q)



0.2 0.1 0.9622056506423076
0.2 0.12000000000000001 0.9623463524068117
0.2 0.14 0.9624715575764877
0.2 0.16000000000000003 0.9625815734891021
0.2 0.18000000000000002 0.962673622339216
0.2 0.2 0.9627439465493502
0.2 0.22000000000000003 0.9627979016388603
0.2 0.24000000000000002 0.962836429209591
0.2 0.26 0.9628583613114187
0.2 0.28 0.9628630737254964
0.2 0.30000000000000004 0.9628535937819326
0.2 0.32000000000000006 0.9628266068943758
0.2 0.3400000000000001 0.9627836574519711
0.2 0.3600000000000001 0.9627219972941659
0.2 0.38 0.9626412836237586
0.2 0.4 0.9625429987191625
0.2 0.42000000000000004 0.9624300393970004
0.2 0.44000000000000006 0.9622990820174931
0.2 0.4600000000000001 0.9621521060679277
0.2 0.4800000000000001 0.9619835641548361
0.21000000000000002 0.1 0.9630953017116576
0.21000000000000002 0.12000000000000001 0.9632360754969231
0.21000000000000002 0.14 0.9633620019821126
0.21000000000000002 0.16000000000000003 0.9634725014086537
0.21000000000000002 0.18000000000000002 0.9635655

In [8]:
plt.figure(figsize=(8,8))
plt.imshow(cor, cmap='gray')

In [7]:
i, j = np.where(cor == np.max(cor))
np.arange(0.2,0.4,0.01)[i], np.arange(0.1,0.5,0.02)[j], cor[i,j]

(array([0.29]), array([0.28]), array([0.96701859]))